In [ ]:
import torch

def softmax_from_scratch(logits):
    """수치적으로 안정적인 Softmax 직접 구현."""
    shifted_logits = (
        logits
        - logits.max(dim=-1, keepdim=True).values
    )
    exponentials = shifted_logits.exp()
    partition = exponentials.sum(dim=-1, keepdim=True)
    return exponentials / partition

In [2]:
class SoftmaxRegressionScratch:
    """Tensor 연산만 사용한 Softmax 회귀 모델."""

    def __init__(
        self,
        num_inputs,  # 입력 차원수: num_inputs
        num_outputs, # 분류 라벨수: num_outputs
        sigma=0.01,
    ):
        self.num_inputs = num_inputs
        self.num_outputs = num_outputs

        # 입력 feature와 출력 class를 연결하는 가중치
        #
        # W.shape = (num_inputs, num_outputs)
        self.weights = torch.normal(
            mean=0.0,
            std=sigma,
            size=(num_inputs, num_outputs),
            requires_grad=True,
        )

        # 클래스마다 하나의 bias를 사용한다.
        #
        # b.shape = (num_outputs,)
        self.bias = torch.zeros(
            num_outputs,
            requires_grad=True,
        )

    def parameters(self):
        """학습할 parameter들을 반환한다."""
        return [
            self.weights,
            self.bias,
        ]

    def forward(self, images):
        # images:
        # (batch_size, channels, height, width)
        #
        # flattened:
        # (batch_size, num_inputs)
        flattened = images.reshape(
            images.shape[0],
            -1,
        )

        if flattened.shape[1] != self.num_inputs:
            raise ValueError(
                "Flattened input size does not match "
                f"num_inputs: {flattened.shape[1]} "
                f"!= {self.num_inputs}"
            )

        # (B, num_inputs) @ (num_inputs, num_outputs)
        # -> (B, num_outputs)
        logits = (
            flattened @ self.weights + self.bias
        )

        # 클래스별 logit을 확률로 변환한다.
        probabilities = softmax_from_scratch(
            logits
        )

        return probabilities

    def __call__(self, images):
        # model.forward(images) 대신
        # model(images)로 호출할 수 있게 한다.
        return self.forward(images)

In [3]:
num_inputs = 32 * 32
num_outputs = 10

model = SoftmaxRegressionScratch(
    num_inputs=num_inputs,
    num_outputs=num_outputs,
)

print("Weights shape:", model.weights.shape)
print("Bias shape:", model.bias.shape)

Weights shape: torch.Size([1024, 10])
Bias shape: torch.Size([10])


In [4]:
# 실제 Fashion-MNIST와 같은 shape의 가상 미니배치
demo_images = torch.rand(
    4,
    1,
    32,
    32,
)

demo_probabilities = model(
    demo_images
)

print("Input shape:", demo_images.shape)
print("Output probabilities:")
print(demo_probabilities)
print("Output shape:", demo_probabilities.shape)

print("\nProbability sums:")
print(demo_probabilities.sum(dim=-1))

Input shape: torch.Size([4, 1, 32, 32])
Output probabilities:
tensor([[0.1074, 0.1338, 0.0838, 0.0853, 0.0856, 0.0962, 0.0855, 0.0829, 0.1449,
         0.0946],
        [0.1128, 0.1017, 0.0906, 0.0932, 0.0820, 0.0737, 0.0787, 0.0938, 0.1473,
         0.1262],
        [0.1311, 0.0931, 0.0899, 0.0775, 0.0704, 0.0911, 0.0832, 0.1098, 0.1434,
         0.1105],
        [0.1424, 0.1055, 0.0777, 0.0795, 0.0786, 0.0811, 0.0954, 0.0828, 0.1609,
         0.0961]], grad_fn=<DivBackward0>)
Output shape: torch.Size([4, 10])

Probability sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [ ]:
def cross_entropy_from_probabilities(
    probabilities,
    labels,
):
    """정답 클래스의 확률을 선택해 평균 Cross-Entropy를 계산한다."""
    
    batch_indices = torch.arange(
        probabilities.shape[0],
        device=probabilities.device,
    )
    
    # 각 데이터에서 정답 클래스의 확률만 선택한다.
    correct_probabilities = probabilities[
        batch_indices, # 배치 인덱스 (ex: [0, 1, 2, 3, ...])
        labels, # 정답 라벨의 인덱스 (ex: [0, 2, 1, 0, 1, ....])
    ]
    
    # log(0)을 방지한 뒤 -log를 적용한다.
    per_example_loss = - (
        correct_probabilities
        .clamp_min(1e-12)
        .log()
    )
    
    return per_example_loss.mean()
    

sample_probabilities = torch.tensor([
    [0.1, 0.3, 0.6],
    [0.3, 0.2, 0.5],
])

sample_labels = torch.tensor([
    0,
    2,
])

sample_loss = cross_entropy_from_probabilities(
    sample_probabilities,
    sample_labels,
)

print("Correct-class probabilities:")
print(sample_probabilities[[0, 1], sample_labels])

print("\nMean Cross-Entropy:")
print(sample_loss)
    
    

Correct-class probabilities:
tensor([0.1000, 0.5000])

Mean Cross-Entropy:
tensor(1.4979)


In [ ]:
def model_loss(
    self,
    probabilities,
    labels,
):
    return cross_entropy_from_probabilities(
        probabilities,
        labels,
    )


# 기존 모델 클래스에 loss 메서드를 추가한다.
SoftmaxRegressionScratch.loss = model_loss


# 위의 Cell 에서 만든 확률을 사용해 loss를 계산한다.
demo_labels = torch.tensor([
    0,
    1,
    2,
    3,
])

demo_loss = model.loss(
    demo_probabilities,
    demo_labels,
)

print("Demo loss:", demo_loss)
print("Loss shape:", demo_loss.shape)

# 배치 손실을 평균냈으므로 scalar다.
assert demo_loss.shape == ()

Demo loss: tensor(2.3644, grad_fn=<MeanBackward0>)
Loss shape: torch.Size([])
